In [1]:
import os
import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.tools import BaseTool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
products = {
    "P1023": {
        "name": "Wireless Mouse",
        "in_stock": True,
        "quantity": 25
    },
    "P1024": {
        "name": "USB Keyboard",
        "in_stock": False,
        "quantity": 0
    },
    "P1025": {
        "name": "Laptop Stand",
        "in_stock": True,
        "quantity": 12
    }
}

print("Sample product inventory created.")
print(f"Number of products: {len(products)}")

Sample product inventory created.
Number of products: 3


In [3]:
class ProductStockInput(BaseModel):
    product_id: str = Field(
        description="Unique identifier of the product"
    )


print("Product stock input schema created.")

Product stock input schema created.


In [4]:
class ProductStockChecker(BaseTool):
    name: str = "check_product_stock"

    description: str = (
        "Check whether a product is in stock using its product ID. "
        "Returns its name, stock status, and available quantity."
    )

    args_schema: type[BaseModel] = ProductStockInput

    def _run(self, product_id: str) -> str:
        """Synchronous product stock lookup."""
        product_id = product_id.strip().upper()
        product = products.get(product_id)

        if product is None:
            return f"No product found for product ID {product_id}."

        status = "In stock" if product["in_stock"] else "Out of stock"

        return (
            f"Product ID: {product_id}\n"
            f"Product name: {product['name']}\n"
            f"Stock status: {status}\n"
            f"Available quantity: {product['quantity']}"
        )

    async def _arun(self, product_id: str) -> str:
        """Asynchronous product stock lookup."""
        return self._run(product_id)


stock_tool = ProductStockChecker()

print("BaseTool subclass created.")
print("Tool name:", stock_tool.name)

BaseTool subclass created.
Tool name: check_product_stock


In [5]:
sync_result = stock_tool.invoke({
    "product_id": "P1023"
})

print("Synchronous result:")
print(sync_result)

assert "P1023" in sync_result
assert "In stock" in sync_result

print("Synchronous test passed.")

Synchronous result:
Product ID: P1023
Product name: Wireless Mouse
Stock status: In stock
Available quantity: 25
Synchronous test passed.


In [6]:
async_result = await stock_tool.ainvoke({
    "product_id": "P1023"
})

print("Asynchronous result:")
print(async_result)

assert "P1023" in async_result
assert "In stock" in async_result

print("Asynchronous test passed.")


Asynchronous result:
Product ID: P1023
Product name: Wireless Mouse
Stock status: In stock
Available quantity: 25
Asynchronous test passed.


In [7]:
print("Synchronous and asynchronous results match:",
      sync_result == async_result)

assert sync_result == async_result

print("Both implementations returned identical results.")

Synchronous and asynchronous results match: True
Both implementations returned identical results.


In [8]:
llm_with_tools = llm.bind_tools([stock_tool])

user_query = "Is product P1023 in stock?"

messages = [HumanMessage(content=user_query)]

response = llm_with_tools.invoke(messages)

print("User query:")
print(user_query)

print("\nModel-suggested tool calls:")
print(response.tool_calls)

if not response.tool_calls:
    raise RuntimeError(
        "The model did not suggest a tool call. Run this cell again."
    )

User query:
Is product P1023 in stock?

Model-suggested tool calls:
[{'name': 'check_product_stock', 'args': {'product_id': 'P1023'}, 'id': 'fc_36509e75-d87e-4a1f-8109-4415d9912c10', 'type': 'tool_call'}]


In [9]:
messages.append(response)
tool_results = []

for call in response.tool_calls:
    if call["name"] != stock_tool.name:
        raise ValueError(f"Unexpected tool: {call['name']}")

    result = stock_tool.invoke(call["args"])
    tool_results.append(result)

    print("Tool:", call["name"])
    print("Arguments:", call["args"])
    print("\nTool result:")
    print(result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

print("\nManual tool execution completed.")

Tool: check_product_stock
Arguments: {'product_id': 'P1023'}

Tool result:
Product ID: P1023
Product name: Wireless Mouse
Stock status: In stock
Available quantity: 25

Manual tool execution completed.


In [10]:
final_response = llm.invoke(messages)

print("Final answer:")
print(final_response.content)

Final answer:
Yes, product **P1023 (Wireless Mouse)** is in stock with **25 units** available.


In [11]:
assert sync_result == async_result
assert response.tool_calls
assert any(
    call["name"] == "check_product_stock"
    for call in response.tool_calls
)
assert tool_results
assert any("In stock" in result for result in tool_results)
assert final_response.content.strip()

print("Validation passed.")
print("Synchronous .invoke() works.")
print("Asynchronous .ainvoke() works.")
print("The model suggested the stock-checking tool.")
print("The tool was executed manually.")
print("The model generated a final answer.")

Validation passed.
Synchronous .invoke() works.
Asynchronous .ainvoke() works.
The model suggested the stock-checking tool.
The tool was executed manually.
The model generated a final answer.


In [12]:
print("Lab 6 - Exercise 5 completed successfully.")
print("Tool method: BaseTool subclass.")
print("Implemented: _run() and _arun().")
print("Tested: .invoke() and .ainvoke().")
print("Completed: bind_tools() workflow.")

Lab 6 - Exercise 5 completed successfully.
Tool method: BaseTool subclass.
Implemented: _run() and _arun().
Tested: .invoke() and .ainvoke().
Completed: bind_tools() workflow.
